# Load and inspect projects using Python API

In [1]:
import logging
logging.basicConfig(
    level=logging.INFO,
    force=True,
)

import warnings
warnings.filterwarnings(action='ignore')

from polymerist.rdutils import disable_kekulized_drawing
disable_kekulized_drawing()

INFO:rdkit:Enabling RDKit 2024.09.5 jupyter extensions
INFO:numexpr.utils:Note: NumExpr detected 64 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
from pathlib import Path
from src.project import (
    PolymerBuildProject,
    load_job_rdmol,
    atoms_validated,
    mechanism_established,
)

# project_path = Path('polyID_test')
project_path = Path('polyID_production')
project = PolymerBuildProject.get_project(project_path)
print(len(project))

[19:06:21] WARNING: not removing hydrogen atom with dummy atom neighbors
INFO:polymerist.smileslib.functgroups:Loading functional group SMARTS data from LUT
INFO:src.reactions:Initializing reaction template 1/8 ("polyester")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 2/8 ("polyamide")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 3/8 ("polyimide")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 4/8 ("polycarbonate_phosgene")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 5/8 ("polycarbonate_nonphosgene")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:Initializing reaction template 6/8 ("polyurethane_isocyanate")
INFO:src.reactions:Initializing test reactants for validation
INFO:src.reactions:I

5056


### Run Jobs

In [ ]:
project.run(
    names=[
        # 'polymerize',
        # 'oligomerize',
        # 'pack_lattice',
        # 'to_interchange',
        'md_export'
    ],
    # jobs=[
    #     project.open_job(id='214e8512b6218c21668ce8c19c6bf359')
    # ]
)

## Mapping results between prior and new results

In [28]:
import pandas as pd

old_project_path = Path('polyID_production_LAMMPS')
old_project = PolymerBuildProject.get_project(old_project_path)
# print(len(old_project))

old_id_map = {(job.doc.smiles_original, job.sp.DOP) : job.id for job in old_project}
new_id_map = {(job.doc.smiles_original, job.sp.DOP) : job.id for job in project}

compatibility_map = {}
num_unmatched : int = 0
for statekey, old_id in old_map.items():
    new_id = new_map.get(statekey)
    compatibility_map[old_id] = new_id
    if new_id is None:
        num_unmatched += 1
print(num_unmatched)

compat_df = pd.DataFrame(compatibility_map.items(), columns=['Pilot set hash', 'Expanded set hash'])
compat_df.to_csv('pilot_to_prod_hashes.csv')

90


## Re-marking sulfur-containing chemistries as eligible for build

In [ ]:
from rich.progress import track
from rdkit.Chem.rdqueries import AtomNumEqualsQueryAtom

sulfur_query = AtomNumEqualsQueryAtom(16, negate=False)

sulfur_hashes : set[str] = set()
for job in track(project, description='Looking for sulfur atoms', total=len(project)):
    mol = load_job_rdmol(job, separate_mols=False)
    if any(mol.GetAtomsMatchingQuery(sulfur_query)):
        sulfur_hashes.add(job.id)

print(len(sulfur_hashes))

In [ ]:
from collections import Counter

c = Counter()
for job_hash in sulfur_hashes:
    job = project.open_job(id=job_hash)
    job.doc.pop('has_banned_atom_types')
    c[atoms_validated(job)] += 1 # verify that this no longer marks these as validated

print(c)

## Checking that aromaticity is being respected

In [ ]:
from src.utils.dataIO import read_rxn_mapping_data
from polymerist.rdutils.reactions.reactions import AnnotatedReaction


RXN_DIR : Path = Path('src/reactions')
# RXN_PATHNAME : str = 'rxn_smarts.json'
RXN_PATHNAME : str = 'rxns_polyID.json'

show : bool = not True

# load reactions
rxns : dict[str, AnnotatedReaction] = {}
for rxnname, smarts in read_rxn_mapping_data(RXN_DIR / RXN_PATHNAME).items():
    rxn = AnnotatedReaction.from_smarts(smarts)
    rxns[rxnname] = rxn 
    if show:
        print(rxnname)
        display(rxn)

In [ ]:
from rdkit import Chem
from rdkit.Chem.rdmolops import SANITIZE_ALL, AROMATICITY_MDL
from polymerist.rdutils.sanitization import sanitize_mol

from polymerist.rdutils.reactions.reactors import PolymerizationReactor
from polymerist.rdutils.reactions.fragment import CutMinimumCostBondsStrategy

build_jobs = [job for job in project if mechanism_established(job)]
job = build_jobs[0]
job = project.open_job(id='455fc6cfdb0fc1e7e8ce8c61ce512567')

monomers = PolymerBuildProject.sanitized_mol_from_smiles(job.sp.smiles_explicit, separate_mols=True)
for m in monomers:
    display(m)

In [ ]:
job = project.open_job(id='57825dbdb6cab389c8f39ad2db3b4c1d')

In [ ]:
import pandas as pd

records : list[dict] = []
for job in project:
    op_times = job.doc.get(PolymerBuildProject.OP_TIME_RECORD_NAME)
    if op_times is not None:
        op_times['Job ID'] = job.id
        records.append(op_times)

## Mapping associated hashes between jobs of same chemistry/DOP jobs across projects

In [1]:
from typing import Optional, Union
from rich.progress import track

import pandas as pd
from src.project import PolymerBuildProject


projnames : tuple[str, str] = 'polyID_production_LAMMPS', 'polyID_production_expanded'
# select smaller of the two projects to map to conserve memory
projects = sorted((
    (projname, PolymerBuildProject.get_project(projname))
        for projname in projnames
    ),
    key=lambda p : len(p[1]),
)
(projname_lesser, project_lesser), (projname_greater, project_greater) = projects

# map statepoint params to lesser project hash
map1 : dict[tuple[str, int], str] = dict()
for job in track(project_lesser, description='Building map of hashes for smaller project'):
    map1[job.doc.smiles_original, job.sp.DOP] = job.id

# recover greater project hashes which corresponds to a lesser's statepoint - collate table
records : list[dict[str, Union[int, str]]] = []
for job in track(project_greater, description='Searching for counterparts in larger project'):
    hash_greater = job.id
    smiles_orig, dop = job.doc.smiles_original, job.sp.DOP
    hash_lesser : Optional[str] = map1.get((smiles_orig, dop), None)

    if hash_lesser is not None:
        records.append({
            'smiles_from_dataset' : smiles_orig,
            'DOP' : dop,
            f'hash in {projname_lesser}' : hash_lesser,
            f'hash in {projname_greater}' : hash_greater,
        })

[23:46:18] WARNING: not removing hydrogen atom with dummy atom neighbors


Output()

Output()

1170


,smiles_from_dataset,DOP,hash in polyID_production_LAMMPS,hash in polyID_production_expanded
0,Nc1ccc(N)cc1.O=C(O)c1cccc(N2C(=O)c3ccc(-c4cccc...,3,7ec2d6317497bf004854b8b8bffb5d1f,ac9b1d35adc0be90b202ee6ee0f37776
1,Nc1ccc(Oc2ccc(Oc3ccc(N)cc3)c3ccccc23)cc1.O=C(O...,3,8af200f7cb6b3343795928166ae786c2,fe8d110b3e5402f9ad22b9bd2e50face
2,O=C(O)CCCCCCCC(=O)O.OCCCCCCCCCCCCCCCCCCCCO,5,52125dad4094b553bc35f6a38b1cefed,c43b9a50082b03478f9d5bab0b05ae7e
3,Nc1cccc2c(Oc3cccc(Oc4cccc5c(N)cccc45)n3)cccc12...,3,0e9d74a00cd1aa55ac677dfe6b544d4c,1f214891dead3659e5384ebb00264fe6
4,OCCO.O=C(O)CCCCCCOc1ccc(C(=O)O)cc1,3,feedc8897f7cf8fba1a7570f93c066ff,13fab722012cb0a4ca73662dc8a387da
5,O=C(O)C(=O)O.NCCCCCCN,3,d45dbac84c4a4545f8f8f07788d3efd0,6cb030b294a7bae58b853f4debaf76e3
6,O=C(O)c1ccc(C(=O)O)cc1.Nc1cccc(Cc2ccc(Oc3ccc(O...,3,a6195d42775855daca0312c892dc835c,fcaf3035381152abfbbf0073a705df77
7,OCCCCCCO.O=C(O)CCCCCCCCCCC(=O)O,5,8df66136b700f9d7a4ca61a4ab69e8fa,fcd7d4c9a5cdf7ce4a4683105a40e889
8,OCCOCCO.O=C(O)CCCCCC(=O)O,3,705c7bc71195e0768e6bf9a786a25f51,04f589811383bc76a8c6d1044d098632
9,O=C(O)CCC=CCCC(=O)O.OCCCCCCCCCCCCO,3,3d1e216e648ac8a93c99eedd6edd3999,d01cf0297f43e821ad412cbcb93d7a2d


OSError: Cannot save file into a non-existent directory: 'ML'

In [3]:
# Save table for reference
df = pd.DataFrame.from_records(records)
print(len(df))
display(df.head(15))
df.to_csv('ML_SMILES/correspondences_LAMMPS_to_expanded.csv', index=False)

1170


,smiles_from_dataset,DOP,hash in polyID_production_LAMMPS,hash in polyID_production_expanded
0,Nc1ccc(N)cc1.O=C(O)c1cccc(N2C(=O)c3ccc(-c4cccc...,3,7ec2d6317497bf004854b8b8bffb5d1f,ac9b1d35adc0be90b202ee6ee0f37776
1,Nc1ccc(Oc2ccc(Oc3ccc(N)cc3)c3ccccc23)cc1.O=C(O...,3,8af200f7cb6b3343795928166ae786c2,fe8d110b3e5402f9ad22b9bd2e50face
2,O=C(O)CCCCCCCC(=O)O.OCCCCCCCCCCCCCCCCCCCCO,5,52125dad4094b553bc35f6a38b1cefed,c43b9a50082b03478f9d5bab0b05ae7e
3,Nc1cccc2c(Oc3cccc(Oc4cccc5c(N)cccc45)n3)cccc12...,3,0e9d74a00cd1aa55ac677dfe6b544d4c,1f214891dead3659e5384ebb00264fe6
4,OCCO.O=C(O)CCCCCCOc1ccc(C(=O)O)cc1,3,feedc8897f7cf8fba1a7570f93c066ff,13fab722012cb0a4ca73662dc8a387da
5,O=C(O)C(=O)O.NCCCCCCN,3,d45dbac84c4a4545f8f8f07788d3efd0,6cb030b294a7bae58b853f4debaf76e3
6,O=C(O)c1ccc(C(=O)O)cc1.Nc1cccc(Cc2ccc(Oc3ccc(O...,3,a6195d42775855daca0312c892dc835c,fcaf3035381152abfbbf0073a705df77
7,OCCCCCCO.O=C(O)CCCCCCCCCCC(=O)O,5,8df66136b700f9d7a4ca61a4ab69e8fa,fcd7d4c9a5cdf7ce4a4683105a40e889
8,OCCOCCO.O=C(O)CCCCCC(=O)O,3,705c7bc71195e0768e6bf9a786a25f51,04f589811383bc76a8c6d1044d098632
9,O=C(O)CCC=CCCC(=O)O.OCCCCCCCCCCCCO,3,3d1e216e648ac8a93c99eedd6edd3999,d01cf0297f43e821ad412cbcb93d7a2d
